In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets, preprocessing, linear_model
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical, plot_model
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive

In [ ]:
# Mount the Google Drive at /content/drive
drive.mount('/content/drive')

# Verify by listing the files in the drive
!ls /content/drive/My\ Drive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 02_20201027_Lab03.pdf
 04_21301122_Experiment_No_02.pdf
'06_20201167_Assignment2 (other).pdf'
 06_20201167_Assignment4.pdf
 11_21301258_Experiment_No_02.pdf
 11_21301258_graded_quiz-1.pdf
 11_21301258_lab_03.pdf
'11_21301258_Md. Ismam Hossain Redwan.pdf'
 166C8609-E959-4723-8E62-87BCBB3FA3C9.jpeg
 1.docx
 1.gdoc
 1-l7uMN3JaKN5hXaRnJLkDt9K9G2RlBVh
'21301258_Md. Ismam Hossain Redwan_21301258_psy101.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment01_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan_CSE422_07_Lab_Assignment02_Turnitin_Summer2024.gdoc'
'21301258_Md. Ismam Hossain Redwan.gdoc'
'321 ass 2.gdoc'
'A1_21301258_Md. Ismam Hossain Redwan_sec-11.pdf'
 Add.gdoc
'after peraphrase.gdoc'
'Amar fasi chai.pdf'
'CamScanner 03-02-2022 22.18 (1).pdf'
'Colab Notebooks'
'Copy of Company links.gsheet'
'Copy of Experiment No 03-Submission F

In [ ]:
import zipfile

# Specify the path to the downloaded ZIP file
zip_file_path = "/content/drive/MyDrive/cse428 project/plantsegv2.zip"

# Specify the directory where you want to extract the dataset
extract_to_directory = "/content/drive/MyDrive/cse428 project"

# Extract the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_directory)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout, Conv2DTranspose)
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import MeanIoU
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import zipfile



# Step 2: Define paths for images and masks
# Update the paths based on the dataset structure
image_base_path = "/content/drive/MyDrive/cse428 project/plantsegv2/images"  # Adjust this path
mask_base_path = "/content/drive/MyDrive/cse428 project/plantsegv2/annotations"    # Adjust this path

# Define subdirectories for train, val, and test splits
subdirs = ["train", "val", "test"]

# Step 3: Function to load images and masks
# This function reads images and corresponding masks, ensuring alignment by filenames
def load_images_and_masks(image_base_path, mask_base_path, subdirs):
    images = []
    masks = []

    for subdir in subdirs:  # Loop through train, val, and test directories
        image_path = os.path.join(image_base_path, subdir)
        mask_path = os.path.join(mask_base_path, subdir)

        if not os.path.exists(image_path):  # Check if image directory exists
            print(f"Image directory does not exist: {image_path}")
            continue
        if not os.path.exists(mask_path):  # Check if mask directory exists
            print(f"Mask directory does not exist: {mask_path}")
            continue

        for img_file in os.listdir(image_path):  # Loop through each image file
            try:
                # Load the image and normalize pixel values to [0, 1]
                img = tf.keras.utils.load_img(os.path.join(image_path, img_file), target_size=(128, 128))
                img = tf.keras.utils.img_to_array(img) / 255.0
                images.append(img)

                # Replace image extension with mask extension to find corresponding mask
                mask_file = img_file.replace(".jpg", ".png")  # Adjust extensions as needed
                mask_full_path = os.path.join(mask_path, mask_file)

                if not os.path.exists(mask_full_path):  # Check if corresponding mask exists
                    print(f"Mask file missing: {mask_full_path}")
                    continue

                # Load the mask and normalize pixel values to [0, 1]
                mask = tf.keras.utils.load_img(mask_full_path, target_size=(128, 128), color_mode="grayscale")
                mask = tf.keras.utils.img_to_array(mask) / 255.0
                masks.append(mask)

            except Exception as e:
                # Handle any errors that occur during file processing
                print(f"Error processing file {img_file}: {e}")

    return np.array(images), np.array(masks)

# Step 4: Load Training, Validation, and Test Data
# Call `load_images_and_masks` for each dataset split
X_train, y_train = load_images_and_masks(image_base_path, mask_base_path, ["train"])
X_val, y_val = load_images_and_masks(image_base_path, mask_base_path, ["val"])
X_test, y_test = load_images_and_masks(image_base_path, mask_base_path, ["test"])

# Step 5: Define the U-Net model
# This is the U-Net architecture for segmentation
def convolution_func(num_filters, input_layer):
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(input_layer)
    x = Dropout(0.1)(x)
    x = Conv2D(num_filters, (3, 3), activation='relu', padding='same')(x)
    return x

def encoder(num_filters, input_layer):
    s = convolution_func(num_filters, input_layer)
    p = MaxPooling2D((2, 2))(s)
    return s, p

def decoder(skip_connection, previous_input, num_filters):
    concatenated_input = concatenate([Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(previous_input), skip_connection])
    output = Conv2DTranspose(num_filters, (3, 3), activation='relu', padding='same')(concatenated_input)
    output = Dropout(0.1)(output)
    output = Conv2DTranspose(num_filters, (3, 3), activation='relu', padding='same')(output)
    return output

def unet_model(input_size=(128, 128, 3)):
    inputs = Input(input_size)

    # Encoder
    s1, p1 = encoder(64, inputs)
    s2, p2 = encoder(128, p1)
    s3, p3 = encoder(256, p2)
    s4, p4 = encoder(512, p3)

    # Bottleneck
    b1 = convolution_func(1024, p4)

    # Decoder
    d1 = decoder(s4, b1, 512)
    d2 = decoder(s3, d1, 256)
    d3 = decoder(s2, d2, 128)
    d4 = decoder(s1, d3, 64)

    # Output
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(d4)

    model = Model(inputs, outputs)
    return model

# Step 6: Compile and Train the Model
model = unet_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), batch_size=16, epochs=25)

# Step 7: Evaluate the Model
# Predict on validation set and compute metrics
predictions = model.predict(X_val)

def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred))

def pixel_accuracy(y_true, y_pred):
    y_pred = (y_pred > 0.5).astype(np.float32)
    return np.mean(y_true == y_pred)

mean_iou = MeanIoU(num_classes=2)
mean_iou.update_state(y_val, predictions)
iou_score = mean_iou.result().numpy()
dice_score = dice_coefficient(y_val, predictions)
pixel_acc = pixel_accuracy(y_val, predictions)

print("Mean IOU:", iou_score)
print("Dice Coefficient:", dice_score)
print("Pixel Accuracy:", pixel_acc)

# Step 8: Visualize Results
for i in range(3):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.title("Input Image")
    plt.imshow(X_val[i])
    plt.subplot(1, 3, 2)
    plt.title("Ground Truth")
    plt.imshow(y_val[i].squeeze(), cmap='gray')
    plt.subplot(1, 3, 3)
    plt.title("Predicted Mask")
    plt.imshow(predictions[i].squeeze(), cmap='gray')
    plt.show()


Epoch 1/25
156/495 ━━━━━━━━━━━━━━━━━━━━ 2:59:27 32s/step - accuracy: 0.7758 - loss: 0.4571